In [ ]:
from eduray import *
from eduray.internals import *
from eduray.visualizer.edu_helpers import *
from eduray.math.helpers import smooth_interpolation, old_perlin_fade
from dataclasses import dataclass
import math
random.seed(42)

COLS = 10
ROWS = 8
RES = 5
SCALE = 0.45

# Noise and procedural textures

In this notebook, we explore several basic noise functions and show how they can be visualized as 2D images. We also demonstrate simple procedural texturing based on these noise functions.

The implemented noise functions are defined in 3D space. Each noise value is computed from a point with coordinates $x$, $y$, and $z$. This is useful for procedural materials, because the same noise function can be evaluated directly at points on object surfaces.

For easier visualization, the noise classes also provide a simplified 2D interface. In this case, the $z$ coordinate is kept constant or ignored, and the noise is sampled only over the $x$ and $y$ coordinates. This allows us to display the result as a regular 2D image and better observe the structure of each noise function.

## White noise

The simplest type of procedural noise used in this notebook is white noise. For each sampled position in space, the noise function returns a pseudo-random value from the interval $[-1, 1]$.

White noise is usually not very useful for texturing on its own, because neighboring values are completely unrelated and the result contains abrupt changes. Nevertheless, it is a good starting point, since it clearly shows the basic idea of assigning deterministic pseudo-random values to spatial positions.


In [ ]:
class WhiteNoise(Noise):
    def value(self, position: Vertex | Vector) -> float:
        x, y, z = position.x, position.y, position.z
        return random.Random(hash((x, y, z))).uniform(-1, 1)

noise = WhiteNoise()
grid = sample_grid(noise.noise_fn)
visualize_noise_2D(grid, title="White Noise")

---
## Simplified Perlin-style gradient noise

Perlin noise produces smoother results than white noise by assigning a pseudo-random
gradient direction to each integer grid corner. These gradients are not sampled
independently for every pixel. Instead, they are chosen deterministically from the
grid coordinates using a permutation table, so the same grid corner always receives
the same gradient.

In this notebook, a simplified 2D variant is used. For didactic clarity, the
implementation chooses gradients from a small fixed set of eight directions. This
makes the algorithm easier to implement and visualize. However, it is not a
historically exact implementation of the original Perlin noise from 1985, where
the gradients were pseudo-random unit vectors. The use of a small predefined set
of gradient directions is closer to the later improved Perlin noise variant.

For each sampled point, the algorithm first finds the grid cell that contains the
point. It then computes the local coordinates inside the cell:

$$
f_x = x - \lfloor x \rfloor,
\qquad
f_y = y - \lfloor y \rfloor.
$$

The four corners of the cell are then evaluated. At each corner, the algorithm
computes a dot product between the corner's gradient direction and the distance
vector from that corner to the sampled point. In the implementation, this is done
by the helper function `_grad2`.

The four dot products are then smoothly interpolated. First, interpolation is
performed in the $x$ direction, and then the two results are interpolated in the
$y$ direction:

$$
N(x, y) =
\operatorname{lerp}
\left(
    \operatorname{lerp}(g_{00}, g_{10}, u),
    \operatorname{lerp}(g_{01}, g_{11}, u),
    v
\right),
$$

where $u$ and $v$ are smoothed versions of the local coordinates $f_x$ and $f_y$.
The smoothing function is the Perlin fade function, which reduces visible
discontinuities between neighboring grid cells.

Compared to white noise, neighboring values are related to each other, so the
resulting pattern is continuous and much smoother. This makes Perlin-style noise
useful for procedural textures such as clouds, marble, terrain, or rough surface
variation.

In [ ]:
# Simplified educational 2D gradient-noise implementation following the
# principles of Perlin noise: permutation hashing, gradient selection,
# fade function, and interpolation.
# Reference: https://cs.nyu.edu/~perlin/noise/
_GRADIENTS = [
    ( 1,  1), (-1,  1), ( 1, -1), (-1, -1),
    ( 1,  0), (-1,  0), ( 0,  1), ( 0, -1),
]


def _grad2(h: int, x: float, y: float) -> float:
    gx, gy = _GRADIENTS[h % len(_GRADIENTS)]
    return gx * x + gy * y


class SimplifiedPerlin2D(Noise):
    def value(self, position: Vertex | Vector) -> float:
        x, y = position.x, position.y

        # Determine the grid cell coordinates
        X = int(math.floor(x)) & 255
        Y = int(math.floor(y)) & 255

        # Compute local coordinates within the cell
        fx = x - math.floor(x)
        fy = y - math.floor(y)

        # Apply the Perlin fade function to the local coordinates
        u = old_perlin_fade(fx)
        v = old_perlin_fade(fy)

        # Hash the grid coordinates to get gradient indices
        A = PERM[X] + Y
        B = PERM[X + 1] + Y

        # Compute the dot products at the corners of the cell
        g00 = _grad2(PERM[A], fx, fy)
        g10 = _grad2(PERM[B], fx - 1, fy)
        g01 = _grad2(PERM[A + 1], fx, fy - 1)
        g11 = _grad2(PERM[B + 1], fx - 1, fy - 1)

        # Perform bilinear interpolation of the dot products
        return lerp(lerp(g00, g10, u), lerp(g01, g11, u), v)

### Visualize the noise

In [ ]:
noise = SimplifiedPerlin2D()
visualize_noise_2D(sample_grid(noise.noise_fn), "Classic Perlin (simplified 2D)")

---
## Improved Perlin noise (2002)

Perlin noise is a procedural noise function designed to produce smooth pseudo-random patterns. Unlike ordinary random noise, where neighbouring values are independent, Perlin noise creates continuous changes between nearby points.

The main improvements are:

- a smoother interpolation curve,
- a fixed set of gradient directions,
- better visual continuity between grid cells,
- fewer visible artifacts compared to the original version.

In the implemented version, space is divided into a regular grid. For each sampled point, the algorithm finds the surrounding grid cell, computes vectors from the cell corners to the sampled point, evaluates dot products with pseudo-random gradient vectors, and smoothly interpolates the results. The output is a continuous pseudo-random value that changes smoothly over space.

In [ ]:
noise = PerlinNoise()
visualize_noise_2D(sample_grid(noise.noise_fn), "Improved Perlin (2002)")

---
## Fractal Brownian Motion (fBM)

Another extension of basic noise is **fBM** (*fractal Brownian motion*). It is not a completely new type of noise function, but rather a way of combining multiple layers of basic noise to create a richer and more detailed structure.

The main idea is to sum several so-called **octaves** of noise. Each additional octave uses a higher frequency and usually a lower amplitude. As a result, the final pattern combines large-scale variation with fine detail and appears more natural than a single layer of noise alone.

In [ ]:
noise = FBMNoise(scale=1.0,strength=2.0,octaves=5,lacunarity=2.0,gain=0.5)
visualize_noise_2D(sample_grid(noise.noise_fn), "FBM Noise")

---
## Turbulence noise

Another useful variation of procedural noise is **turbulence noise**. It is based on the same general idea as fBM, but instead of directly summing the signed noise values, it uses their absolute values before combining them. This folds negative parts of the signal upward and produces a pattern with sharper, more vein-like or cloud-like structures.


In [ ]:
noise = TurbulenceNoise(scale=1.0, strength=2.0, octaves=5, lacunarity=1.5, gain=0.5)
visualize_noise_2D(sample_grid(noise.noise_fn), "Turbulence Noise")

---
## Procedural textures

In this part, we create a simple procedural material that gives an object a more natural, rock-like appearance.

Instead of using an image texture, which would require UV coordinates and texture mapping, the material computes its color directly from the position of the hit point on the surface. This makes the material easier to apply to different objects, because the texture is generated mathematically rather than read from an image.

The material is based on `PhongMaterial`, but it overrides the `sample()` method. This method returns a `PhongMaterialSample`, which contains the actual material parameters used by the shading model at the current surface point.

The important idea is that the material does not have one fixed color. For every intersection point, a noise function is evaluated and its value is used to interpolate between a darker and a lighter version of the base color. As a result, different parts of the same object can have slightly different colors, creating a more irregular and natural-looking surface.

In [ ]:
@dataclass
class RockMaterial(PhongMaterial):
    color_scale: float = 3.0
    color_noise: Noise | None = None
    normal_noise: Noise | None = None

    def sample(self, hit: SurfaceInteraction) -> PhongMaterialSample:
        p = hit.point

        if self.color_noise is not None:
            t = 0.5 * self.color_noise.value(p * Vertex(self.color_scale, self.color_scale, self.color_scale)) + 0.5
            t = smooth_interpolation(0.35, 0.75, t)
        else:
            t = 0.5

        dark = self.base_color * 0.2
        light_color = self.base_color * 1.5
        base_color = dark * (1.0 - t) + light_color * t
        ambient_color = self.base_color * 0.5

        shin = self.shininess * (1.0 - t)

        return PhongMaterialSample(
            base_color=base_color,
            spec_color=self.spec_color,
            shininess=float(shin),
            ior=self.ior,
            transparency=self.transparency,
            normal_noise=self.normal_noise,
            ambient_color=ambient_color,
        )

## Rock material and test scene

To demonstrate the effect of procedural texturing and normal perturbation, we define a helper function `make_rock_material()` that creates a rock-like material with optional color variation and optional surface detail.

In [ ]:
def make_rock_material(use_texture: bool, use_noise: bool) -> RockMaterial:
    return RockMaterial(
        base_color=Color.custom_rgb(110, 90, 90),
        spec_color=Color.custom_rgb(200, 210, 210),
        shininess=8,
        color_noise=FBMNoise(
            scale=2.0,
            strength=1.0,
            octaves=5,
            lacunarity=2.0,
            gain=0.5,
        ) if use_texture else None,
        normal_noise=TurbulenceNoise(
            scale=2.0,
            strength=0.5,
            octaves=5,
            lacunarity=2.0,
            gain=0.5,
        ) if use_noise else None,
        color_scale=3.0,
    )

def make_scene(material: Material) -> Scene:
    rock_sphere = Object(
        Sphere(),
        material,
    ).translate(0, 0, 1)

    return Scene(
        camera=PinholeCamera(
            fov_deg=40,
            origin=Vertex(0, 0, 5),
            direction=Vector(0, 0, -1),
        ),
        objects=[rock_sphere],
        lights=[
            PointLight(position=Vertex(5, 8, 9), intensity=0.35),
            PointLight(position=Vertex(-5, 8, 9), intensity=0.35),
            AmbientLight(intensity=0.2),
        ],
        skybox="black"
    )

## Rendering the comparison

The following code renders four versions of the same object in order to compare the effect of procedural texture and normal perturbation.


In [ ]:
scenarios = [
    ("images/noise_none.png", False, False),
    ("images/noise_texture.png", True, False),
    ("images/noise_normal.png", False, True),
    ("images/noise_procedural_texture.png", True, True),
]

for filename, texture, noise in scenarios:
    material = make_rock_material(use_texture=texture, use_noise=noise)
    scene = make_scene(material)

    render_loop = LinearRenderLoop(
        scene=scene,
        render_config=RenderConfig(
            resolution=Resolution.custom(300, 300),
            samples_per_pixel=3,
            max_depth=3,
        ),
        preview_config=PreviewConfig(
            progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW
        ),
        # shading_model=NormalShader()
    )

    image = render_loop.render(filename)
    ipynb_display_images(image)


## Final scene with procedural materials

To demonstrate the procedural rock material in a more complete setting, we place it into a simple scene together with a marble-like background and render the result with colored lighting with falloff.


In [ ]:
marble = MarbleMaterial(
    name="marble",
    base_color=Color.custom_rgb(220, 220, 220),
    spec_color=Color.custom_rgb(255, 255, 255),
    color_one_factor= 0.1,
    color_two_factor= 1.0,
    shininess=20,
    reflectivity=0.25,
)

rock_object = Object(
    geometry=Sphere(),
    material=make_rock_material(use_texture=True, use_noise=True),
).translate(0, 0, 1)

marble_object = Object(
    geometry=Plane(point=Vertex(-5, 0, 0), normal=Vector(0, 0, 1)),
    material=marble,
)

marble_scene = Scene(
        camera=PinholeCamera(
            fov_deg=40,
            origin=Vertex(0, 0, 5),
            direction=Vector(0, 0, -1),
        ),
        objects=[rock_object, marble_object],
        lights=[
            AmbientLight(intensity=0.2),
            PointLightFalloff(position=Vertex(5, 8, 9), intensity=60, color=Color.custom_rgb(150, 180, 250)),
            PointLightFalloff(position=Vertex(-5, 8, 9), intensity=60, color=Color.custom_rgb(250, 180, 150)),
        ],
        skybox="black"
)

rt = LinearRenderLoop(
        scene=marble_scene,
        render_config=RenderConfig(
            resolution=Resolution.R360p,
            samples_per_pixel=1,
            max_depth=1,
        ),
        preview_config=PreviewConfig(
            progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW
        ),
        # shading_model=NormalShader()
    )

image = rt.render("images/textures.png")
ipynb_display_images(image)


# Summary

In this notebook, we explored procedural noise and showed how it can be used to create simple procedural textures for ray tracing. The noise functions were first visualized as 2D images, which makes it easier to understand their structure before using them inside materials.

We started with white noise and classic Perlin noise, then moved to Improved Perlin noise and fractal variants such as fBM and turbulence noise. After that, we used these noise functions in a rock-like material, where noise controls both color variation and normal perturbation.

- **White noise** — a simple pseudo-random function where neighbouring values are unrelated, useful mainly as a basic starting point
- **Classic Perlin noise** — smoother gradient noise based on interpolated dot products at grid corners
- **Improved Perlin noise** — a refined version of Perlin noise with smoother interpolation and fewer visible artifacts
- **fBM noise** — multiple octaves of noise combined together to create richer patterns with both large-scale and fine detail
- **Turbulence noise** — a fractal noise variant based on absolute noise values, producing sharper and more folded structures
- **Procedural texture mapping** — computing material properties directly from the surface hit position instead of using image textures and UV coordinates
- **Rock material** — a `PhongMaterial` extension that evaluates noise in `sample()` and uses it to interpolate between darker and lighter color variations
- **Normal perturbation** — using procedural turbulence noise to modify the shading normal and simulate small surface irregularities without changing the actual geometry
- **Material comparison** — rendering four variants of the same sphere: without noise, with color texture only, with normal perturbation only, and with both effects enabled
- **Final textured scene** — combining the procedural rock material with a marble-like material and multiple colored lights to demonstrate the effect in a more complete scene

These experiments show that procedural noise can add visible surface detail and natural variation while keeping the geometry simple. This is useful in a didactic ray tracer because the effect can be demonstrated without requiring image textures, UV mapping, or complex asset preparation.

---
## References and further reading:
- Perlin, K. (1985). An Image Synthesizer.
- Perlin, K. (2002). Improving Noise.
- Blinn, J. F. (1978). Simulation of Wrinkled Surfaces.
- Ebert, D. S., Musgrave, F. K., Peachey, D., Perlin, K. & Worley, S. (2003).
  *Texturing and Modeling: A Procedural Approach* (3rd ed.).
- Pharr, M., Jakob, W. & Humphreys, G. (2016). *Physically Based Rendering: From Theory to Implementation* (3rd ed.).
  See also https://www.pbr-book.org/ for the online 4th edition.
- Shirley, P., Black, T. D. & Hollasch, S. (2025).
  *Ray Tracing in One Weekend*. https://raytracing.github.io/